# Calories Burnt Prediction Using Machine Learning
### Supervised Regression Pipeline for Fitness and Energy Expenditure Modeling

**Objective:** Build an end-to-end predictive machine learning model to estimate the number of calories burned during physical exercise based on biometric data (gender, age, height, weight) and physiological workout parameters (duration, heart rate, body temperature).

**Models Compared:**
1. Linear Regression
2. Random Forest Regressor
3. XGBoost Regressor

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%matplotlib inline
sns.set_theme(style="whitegrid")

## 1. Data Ingestion & Preprocessing

In [2]:
data_path = "../data/calories_data.csv" if os.path.exists("../data/calories_data.csv") else "data/calories_data.csv"
df = pd.read_csv(data_path)

print(f"Dataset Shape: {df.shape}")
df.head()

In [3]:
# Encode Gender (male: 0, female: 1)
if df["Gender"].dtype == object:
    df["Gender"] = df["Gender"].map({"male": 0, "female": 1})

# Check summary statistics and missing values
print("Missing Values:\n", df.isnull().sum())
df.describe()

## 2. Exploratory Data Analysis & Feature Correlation

In [4]:
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Correlation Heatmap - Biometric & Calorie Attributes")
plt.show()

## 3. Train-Test Split (80% Train / 20% Test)

In [5]:
drop_cols = ["User_ID", "Calories"] if "User_ID" in df.columns else ["Calories"]
X = df.drop(columns=drop_cols)
y = df["Calories"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

## 4. Model Training & Comparative Evaluation

In [6]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42)
}

try:
    from xgboost import XGBRegressor
    models["XGBoost Regressor"] = XGBRegressor(n_estimators=200, learning_rate=0.08, random_state=42)
except ImportError:
    pass

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    results[name] = {"MAE": mae, "RMSE": rmse, "R2": r2, "preds": preds}
    print(f"{name:25s} | MAE: {mae:.3f} kcal | RMSE: {rmse:.3f} kcal | R2: {r2:.4f}")

## 5. Residual Analysis & Actual vs. Predicted Plots

In [7]:
best_model_name = max(results, key=lambda k: results[k]["R2"])
best_preds = results[best_model_name]["preds"]

plt.figure(figsize=(8, 6))
plt.scatter(y_test, best_preds, alpha=0.5, color="#2563eb", edgecolors="w")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2)
plt.xlabel("Actual Calories (kcal)")
plt.ylabel("Predicted Calories (kcal)")
plt.title(f"Actual vs. Predicted - {best_model_name} (R2: {results[best_model_name]['R2']:.4f})")
plt.show()

## 6. Sample Inference

In [8]:
# Test: Male, Age 28, Height 178cm, Weight 75kg, 30 min duration, 120 bpm, 39.5 C body temp
sample_user = np.array([[0, 28, 178.0, 75.0, 30.0, 120.0, 39.5]])
pred_cal = models[best_model_name].predict(sample_user)[0]
print(f"Predicted Calories Burnt for 30-min workout: {pred_cal:.2f} kcal")